In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load files

## Load target files

In [6]:
df_test = pd.read_csv("../data/raw/price_per_square_meter_2020.csv")
df_test.head()

,,,,,,"13mu -- Prices per square meter of old dwellings in housing companies and numbers of transactions by postal code area, yearly, 2009-2025"
Postal code,"Price per square meter (EUR/m2) 2020 Blocks of flats, one-room flat","Price per square meter (EUR/m2) 2020 Blocks of flats, two-room flat","Price per square meter (EUR/m2) 2020 Blocks of flats, three-room flat+","Number of sales, asset transfer tax data starting from 2020 2020 Blocks of flats, one-room flat","Number of sales, asset transfer tax data starting from 2020 2020 Blocks of flats, two-room flat","Number of sales, asset transfer tax data start..."
00100 Helsinki keskusta - Etu-Töölö (Helsinki),8419,7773,7153,110,178,195
00120 Punavuori - Bulevardi (Helsinki),8702,7993,8065,55,82,75
00130 Kaartinkaupunki (Helsinki),...,8588,8083,5,11,19
00140 Kaivopuisto - Ullanlinna (Helsinki),8948,8416,8811,73,69,92


In [7]:
def load_year(path, year):
    df = pd.read_csv(path, skiprows=2, encoding="utf-8-sig")
    
    # rename the 6 data columns to something simple
    df.columns = ["postal_raw", "p1", "p2", "p3", "n1", "n2", "n3"]
    
    # split out prices and sales separately
    prices = df.melt(
        id_vars="postal_raw",
        value_vars=["p1", "p2", "p3"],
        var_name="rooms",
        value_name="price_per_sqm",
    )
    sales = df.melt(
        id_vars="postal_raw",
        value_vars=["n1", "n2", "n3"],
        var_name="rooms",
        value_name="n_sales",
    )
    
    # turn p1/n1 into 1, p2/n2 into 2, etc.
    prices["rooms"] = prices["rooms"].str[1].astype(int)
    sales["rooms"] = sales["rooms"].str[1].astype(int)
    
    # put them back together
    out = prices.merge(sales, on=["postal_raw", "rooms"])
    out["year"] = year
    
    return out

In [8]:
years = [2020, 2021, 2022, 2023, 2024]

df = pd.concat(
    [load_year(f"../data/raw/price_per_square_meter_{y}.csv", y) for y in years],
    ignore_index=True,
)

In [9]:
df["postal_code"]  = df["postal_raw"].str[:5]
df["municipality"] = df["postal_raw"].str.extract(r"\((.*?)\)$")

df["price_per_sqm"] = pd.to_numeric(df["price_per_sqm"], errors="coerce")
df["n_sales"]       = pd.to_numeric(df["n_sales"], errors="coerce")

df = df[["postal_code", "municipality", "year", "rooms", "price_per_sqm", "n_sales"]]

In [10]:
df["status"] = "published"
df.loc[df.price_per_sqm.isna() & df.n_sales.notna(), "status"] = "suppressed"
df.loc[df.n_sales.isna(), "status"] = "no_sales"

In [11]:
df.head()

,postal_code,municipality,year,rooms,price_per_sqm,n_sales,status
0,00100,Helsinki,2020,1,8419.0,110.0,published
1,00120,Helsinki,2020,1,8702.0,55.0,published
2,00130,Helsinki,2020,1,NaN,5.0,suppressed
3,00140,Helsinki,2020,1,8948.0,73.0,published
4,00150,Helsinki,2020,1,8739.0,134.0,published


## Load features files

In [12]:
df_f_test = pd.read_csv('../data/raw/average_floor_area_per_dwelling_2020_2024.csv', skiprows=1)
df_f_test.head()

,Postal code area,2020 Average floor area per dwelling (RA),2021 Average floor area per dwelling (RA),2022 Average floor area per dwelling (RA),2023 Average floor area per dwelling (RA),2024 Average floor area per dwelling (RA)
0,00100 Helsinki keskusta - Etu-Töölö (Helsinki),65.8,65.8,65.7,65.6,65.7
1,00120 Punavuori - Bulevardi (Helsinki),69.5,69.6,69.8,70.0,70.0
2,00130 Kaartinkaupunki (Helsinki),78.5,78.4,79.3,80.2,81.0
3,00140 Kaivopuisto - Ullanlinna (Helsinki),74.1,74.1,74.4,74.2,74.4
4,00150 Punavuori - Eira - Hernesaari (Helsinki),55.9,55.9,56.2,56.6,56.7


In [13]:
paavo_files = {
    "population":        "../data/raw/population_structure_2020_2024.csv",
    "age18plus":         "../data/raw/age_18_or_over_2020_2024.csv",
    "higher_degree":     "../data/raw/academic_degree_higher_university_level_degree_2020_2024.csv",
    "income":            "../data/raw/average_income_of_households_2020_2024.csv",
    "hh_size":           "../data/raw/average_size_of_households_2020_2024.csv",
    "hh_total":          "../data/raw/households_total_2020_2024.csv",
    "hh_rented":         "../data/raw/households_living_in_rented_dwellings_2020_2024.csv",
    "hh_one_person":     "../data/raw/one_person_households_2020_2024.csv",
    "dwellings":         "../data/raw/dwellings_2020_2024.csv",
    "dwellings_flats":   "../data/raw/dwellings_in_blocks_of_flats_2020_2024.csv",
    "avg_floor_area":    "../data/raw/average_floor_area_per_dwelling_2020_2024.csv",
    "unemployed":        "../data/raw/unemployment_new_2020_2024.csv",
    "workplaces":        "../data/raw/workplaces_total_2020_2024.csv",
}

In [14]:
def load_paavo(path, name):
    d = pd.read_csv(path, skiprows=2, encoding="utf-8-sig")
    d = d.rename(columns={d.columns[0]: "postal_raw"})
    
    long = d.melt(id_vars="postal_raw", var_name="col", value_name=name)
    long["year"] = long["col"].str.extract(r"(20\d{2})").astype(int)
    long["postal_code"] = long["postal_raw"].str[:5]
    long[name] = pd.to_numeric(long[name], errors="coerce")
    
    return long[["postal_code", "year", name]]

In [15]:
from functools import reduce

frames = [load_paavo(p, n) for n, p in paavo_files.items()]

paavo = reduce(
    lambda a, b: a.merge(b, on=["postal_code", "year"], how="outer"),
    frames
)

paavo.loc[paavo.year == 2024, "workplaces"] = pd.NA

In [16]:
print(paavo.shape)        # expect ~15,000 rows (3018 × 5 years)
paavo.head()

(15090, 15)


,postal_code,year,population,age18plus,higher_degree,income,hh_size,hh_total,hh_rented,hh_one_person,dwellings,dwellings_flats,avg_floor_area,unemployed,workplaces
0,00100,2020,18373,16236,6023.0,70284.0,1.7,10380,5289.0,5379.0,12341.0,11721.0,65.8,1083.0,49712.0
1,00100,2021,17893,15817,5949.0,73553.0,1.7,10141,5157.0,5267.0,12321.0,11663.0,65.8,723.0,52491.0
2,00100,2022,18030,15997,5977.0,70016.0,1.7,10280,5334.0,5447.0,12337.0,11719.0,65.7,630.0,55404.0
3,00100,2023,18462,16365,6225.0,70761.0,1.7,10568,5527.0,5620.0,12440.0,11819.0,65.6,693.0,55646.0
4,00100,2024,18492,16397,6422.0,73915.0,1.7,10544,5352.0,5546.0,12469.0,11869.0,65.7,834.0,NaN


In [17]:
data = (df
        .merge(paavo, on=["postal_code", "year"], how="left"))

In [18]:
data.head()

,postal_code,municipality,year,rooms,price_per_sqm,n_sales,status,population,age18plus,higher_degree,income,hh_size,hh_total,hh_rented,hh_one_person,dwellings,dwellings_flats,avg_floor_area,unemployed,workplaces
0,00100,Helsinki,2020,1,8419.0,110.0,published,18373.0,16236.0,6023.0,70284.0,1.7,10380.0,5289.0,5379.0,12341.0,11721.0,65.8,1083.0,49712.0
1,00120,Helsinki,2020,1,8702.0,55.0,published,7216.0,6221.0,2275.0,74856.0,1.8,4047.0,1840.0,2042.0,4777.0,4492.0,69.5,460.0,7131.0
2,00130,Helsinki,2020,1,NaN,5.0,suppressed,1721.0,1501.0,571.0,101682.0,1.8,948.0,460.0,445.0,1192.0,1000.0,78.5,81.0,11708.0
3,00140,Helsinki,2020,1,8948.0,73.0,published,7995.0,6872.0,2595.0,92033.0,1.8,4497.0,2102.0,2263.0,5391.0,5358.0,74.1,465.0,1919.0
4,00150,Helsinki,2020,1,8739.0,134.0,published,9440.0,8322.0,2857.0,59378.0,1.6,5798.0,2799.0,3424.0,6694.0,6616.0,55.9,682.0,4176.0


In [19]:
data.isnull().sum()

postal_code            0
municipality           0
year                   0
rooms                  0
price_per_sqm      19654
n_sales            14230
status                 0
population            15
age18plus             15
higher_degree         69
income               162
hh_size              162
hh_total              15
hh_rented            162
hh_one_person        162
dwellings             30
dwellings_flats       30
avg_floor_area        30
unemployed            54
workplaces          5184
dtype: int64

In [20]:
data.notna().all(axis=1).sum()

np.int64(5058)

In [21]:
data.shape

(25860, 20)

In [22]:
data.duplicated(subset=['postal_code', 'year']).sum()

np.int64(17240)